# 如何链式运行可运行对象
- 关于`LangChain表达`式的一点是，任何两个可运行对象可以“链式”组合成序列。前一个可运行对象的 .invoke() 调用的输出作为输入传递给下一个可运行对象。
    - 这可以使用`管道操作符 (|) `
    - 或更明确的 `.pipe()` 方法来完成

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("apikey.env")
BASE_URL = 'https://api.deepseek.com'
API_KEY = os.getenv('DEEPSEEK-API-KEY')
deepseek_chat_model = 'deepseek-chat'
if  not API_KEY:
    raise ValueError("WARNING: NOT FOUND OPENAI_API_KEY，PLEASE CHECK .env SETING。")
else:
    print("SECESSFULLY!")

SECESSFULLY!


## 管道操作符 `|`
例子使用提示词模板将输入格式化为聊天模型，最后将聊天消息输出转换为字符串，使用输出解析器。

In [11]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_deepseek import ChatDeepSeek

llm = ChatDeepSeek(api_key=API_KEY, base_url=BASE_URL, model=deepseek_chat_model)

prompt = ChatPromptTemplate.from_template("给我说一个{topic}的笑话")
chain = prompt | llm | StrOutputParser()

chain.invoke({"topic" : "鸭子"})

'当然！这里有一个经典的鸭子笑话：\n\n---\n\n一只鸭子走进一家便利店，摇摇摆摆地走到柜台前，对店员说：“你们这儿有葡萄吗？”\n\n店员有点惊讶，但还是礼貌地说：“不好意思，我们没有葡萄。”\n\n鸭子听了，说了声“谢谢”，然后就走了。\n\n第二天，鸭子又来了，同样走到柜台前问：“你们这儿有葡萄吗？”\n\n店员有点不耐烦了，说：“没有！我们昨天没有，今天也没有！如果你再来问葡萄，我就用胶带把你的嘴粘上！”\n\n鸭子点点头，又走了。\n\n第三天，鸭子再次走进店里，径直走到柜台前，看着店员问：“你们这儿有胶带吗？”\n\n店员愣了一下，说：“没有，我们这儿不卖胶带。”\n\n鸭子立刻追问：“那——你们这儿有葡萄吗？”\n\n---\n\n希望这个笑话能让你笑一笑！😄'

## 强制转换
我们甚至可以将这个链与更多的可运行项结合起来，创建另一个链。

In [20]:
from langchain_core.output_parsers import StrOutputParser

analysis_prompt = ChatPromptTemplate.from_template("这个{joke}好笑程度你给几分，满分10分。")
composed_chain = {"joke" : chain} | analysis_prompt | llm | StrOutputParser()

async for chunk in chain.astream({"topic": "周末"}):
    print(chunk, end="", flush=True)
async for chunk in composed_chain.astream({"topic": "周末"}):
    print(chunk, end="", flush=True)

当然！说一个关于周末的“摸鱼”笑话：

---

周末加班，老板突然来办公室巡视，看见小张正对着电脑眉头紧锁。  
老板欣慰地拍拍他肩膀：“辛苦啦！周末还在努力！”  
小张吓得一抖，慌忙关掉屏幕说：“应该的！在赶项目进度！”  

老板走后，同事小声问：“你刚才到底在干嘛？”  
小张崩溃捂脸：“……我在查‘周末加班如何假装在干活’。”  

---

祝你周末愉快，不用“假装”干活！ 😄哈哈，这个笑话我给**8.5分**！  
理由：  
- **反差萌**：从“人生哲学”瞬间跌回“剥蒜日常”，理想与现实的碰撞特别有生活气息。  
- **共鸣感**：周末本想躺平思考诗和远方，结果被拉回厨房战场，打工人秒懂！  
- **老婆的犀利**：一句“为了不用思考人生的周末”堪称灵魂暴击，幽默里藏着真理。  

**扣分点**：如果老公回一句“蒜了吧，人生需要朦胧美”，或许能再加0.5分冷笑话buff～  

您的周末版「人间清醒」已送达，记得多放松、少剥蒜！ 😉

In [22]:
composed_chain_with_lambda = (
    chain
    | (lambda input: {"joke": input})
    | analysis_prompt
    | llm
    | StrOutputParser()
)

async for chunk in composed_chain_with_lambda.astream({"topic": "野兽"}):
    print(chunk, end="", flush=True)

哈哈，这个笑话我给**8.5分**！🦁🐰  
**加分点**：  
1. **反差萌**——狮子威猛的咆哮 vs 兔子细弱的“吱吱”，瞬间戳中笑点；  
2. **神转折结局**——“别人听了只会想给你胡萝卜”既贴合兔子形象，又自带吐槽属性；  
3. **画面感超强**——脑补小兔子认真“咆哮”的样子，简直可爱到犯规！  

**扣分点**：  
稍微有点“经典套路”，但结尾的幽默救回来了！如果改成狮子默默递上一根胡萝卜，可能分数更高哦～ 😂  
你心中的分数是多少呀？欢迎继续挑战更野的野兽笑话！

# 2. pipe()方法


In [28]:
from langchain_core.runnables import RunnableParallel

composed_chain_with_pipe = (
    RunnableParallel({"joke": chain})
    .pipe(analysis_prompt)
    .pipe(llm)
    .pipe(StrOutputParser())
)

async for chunk in chain.astream({"topic": "你好"}):
    print(chunk, end="", flush=True)
print("\n=======================\n")
async for chunk in composed_chain_with_pipe.astream({"topic": "你好"}):
    print(chunk, end="", flush=True)

当然！这里有一个关于“你好”的简单笑话：

---

有一天，一个英文单词“Hello”走在路上，突然被一块石头绊倒了。

它爬起来后，很生气地对着石头说：**“哎哟！你干嘛撞我？”**

石头一脸无辜地回答：**“我……我不认识你啊！”**

“Hello”愣了一下，突然反应过来：**“哦对，我忘了自我介绍——你好，我是‘Hello’！”**

---

希望这个谐音梗能让你会心一笑！😄 如果需要其他类型的笑话，我随时等你来问～

哈哈，这个笑话我给7.5分！  
**加分点**：  
- 语言误解造成的冲突很生活化，容易让人会心一笑。  
- 结尾的“我还‘Goodbye’你呢”突然切换英文，意外又有趣，像一场跨频道吵架。  

**扣分理由**：  
- 算是经典“语言梗”模板，稍微有点老套～  
不过整体节奏轻快，放在中文学习场景里依然可爱！  

要不要再来一个谐音梗笑话？比如“为什么西瓜不说话？” 😄